# Comparison Corpora: Exploratory Analysis

This notebook is set up for exploring the three comparison datasets: `r/conspiracy_commons`, `r/AskReddit`, and `r/TopMindsOfReddit`.

The pre-scored parquets are already loaded. The goal here is to **read the comments**, check if the lexical dimensions are capturing the intended epistemic moves in these different environments, and decide how to frame these subreddits in the final thesis.

In [ ]:
import duckdb
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', None)
con = duckdb.connect()

In [ ]:
# Load the pre-scored parquets into memory
df_commons = con.execute("SELECT * FROM read_parquet('../data/processed/comparison_conspiracy_commons_scored.parquet')").df()
df_askreddit = con.execute("SELECT * FROM read_parquet('../data/processed/comparison_askreddit_scored.parquet')").df()
df_topminds = con.execute("SELECT * FROM read_parquet('../data/processed/comparison_topmindsofreddit_scored.parquet')").df()

print(f"Loaded:\nCommons: {len(df_commons):,} comments\nAskReddit: {len(df_askreddit):,} comments\nTopMinds: {len(df_topminds):,} comments")

## 1. r/conspiracy_commons Exploration
Is this a 'purer' version of the main sub, or something else entirely?

In [ ]:
# Let's look at what highly upvoted comments in Commons look like
df_commons.sort_values('upvotes', ascending=False)[['upvotes', 'text', 'evidence_count', 'hedge_count']].head(10)

In [ ]:
# Let's inspect the 'Adversarial' rhetoric, since it averaged 3.91 upvotes here
df_commons[df_commons['adversarial_count'] > 0][['upvotes', 'text']].sort_values('upvotes', ascending=False).head(10)

## 1.1 Deep Dive: Top Comments per Dimension
Let's look at the highest upvoted comments for each individual lexical dimension to see what they capture in `commons`.

In [ ]:
from IPython.display import display

dimensions = ['evidence_count', 'adversarial_count', 'hedge_count', 'certainty_count', 'alt_authority_count', 'intuitive_count', 'pattern_count', 'meta_count', 'demand_count', 'anecdotal_count', 'quantitative_count']

for dim in dimensions:
    print(f"\n=== Top comments for {dim} ===")
    top_comments = df_commons[df_commons[dim] > 0].sort_values('upvotes', ascending=False)[['upvotes', 'text']].head(3)
    display(top_comments)


## 1.2 Dimension Co-occurrence (Correlation Matrix)
Do people who cite evidence also demand proof? Do people who use adversarial rhetoric also use hedges?

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

dimensions = ['evidence_count', 'adversarial_count', 'hedge_count', 'certainty_count', 'alt_authority_count', 'intuitive_count', 'pattern_count', 'meta_count', 'demand_count', 'anecdotal_count', 'quantitative_count']

# Calculate correlation matrix
corr = df_commons[dimensions].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-0.1, vmax=1.0)
plt.title('Correlation Matrix of Lexical Dimensions in r/conspiracy_commons')
plt.tight_layout()
plt.show()


## 2. r/AskReddit Exploration
The baseline control. We know 'Stats/Data' gets rewarded here. What does that actually look like?

In [ ]:
df_askreddit[df_askreddit['quantitative_count'] > 0].sort_values('upvotes', ascending=False)[['upvotes', 'text']].head(10)